<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week04_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def get_rmse(R, P, Q, non_zeros):
    error = 0
    for i, j, r in non_zeros:
        pred = np.dot(P[i, :], Q[j, :].T)
        error += pow(r - pred, 2)
    rmse = np.sqrt(error / len(non_zeros))
    return rmse

> 파이썬 머신러닝 완벽 가이드 9장 p.625~647

## 9-07 행렬 분해를 이용한 잠재 요인 협업 필터링 실습

- SVD (일반적)
  - SGD, ALS  
  : 사용자-아이템 평점 행렬처럼 NaN 데이터 많은 경우 사용.
- NMF



> 행렬 분해 로직 함수 생성
- R: 원본 사용지一아이템 평점 행렬
- K: 잠재 요인의 차원 수
- steps: SGD의 반복 횟수
- learningjrate: 학습률
- r_lambda: L2 규제 계수


In [ ]:
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01):
  num_users, num_items = R.shape
  # P와 Q 매트릭스의 크기를 지정하고 정규 분포를 가진 랜덤한 값으로 입력
  np.random.seed(1)
  P = np.random.normal(scale=1./K, size=(num_users, K))
  Q = np.random.normal(scale=1./K, size=(num_items, K))

  # R > 0 인 행 위치, 열 위치, 값을 non_zeros 리스트 객체에 저장.
  non_zeros = [ (i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i,j] > 0 ]

  # SGD 기법으로 P와 Q 매트릭스를 계속 업데이트.
  for step in range(steps):
    for i, j, r in non_zeros:
      # 실제 값과 예측 값의 차이인 오류 값 구함
      eij = r - np.dot(P[i,:], Q[j,:].T)
      # Regularization을 반영한 SGD 업데이트 공식 적용
      P[i, :] = P[i, :] + learning_rate*(eij * Q[j, :] - r_lambda*P[i, :])
      Q[j, :] = Q[j, :] + learning_rate*(eij * P[i, :] - r_lambda*Q[j, :])

    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 10) == 0:
      print('### iteration step : ', step , 'rmse :', rmse)

  return P, Q

>  영화 평점 행렬 데이터를 DataFrame으로 로딩한 후 다시 사용자-아이템 평점 행렬로 변경

In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv('/content/movies.csv')
ratings = pd.read_csv('/content/ratings.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
ratings_matrix = ratings.pivot_table('rating', index='userId', columns='movieId')

# title 칼럼을 얻기 위해 movies와 조인 수행
rating_movies = pd.merge(ratings, movies, on='movieId')

# columns='title'로 title 칼럼으로 pivot 수행
ratings_matrix = rating_movies.pivot_table('rating', index='userId', columns='title')

> 생성한 함수로 행렬 분해

In [ ]:
P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01,
                            r_lambda = 0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0 rmse : 2.9023619751337115
### iteration step :  10 rmse : 0.7335768591017939
### iteration step :  20 rmse : 0.5115539026853438
### iteration step :  30 rmse : 0.37261628282537734
### iteration step :  40 rmse : 0.29608182991810145
### iteration step :  50 rmse : 0.2520353192341621
### iteration step :  60 rmse : 0.22487503275269882
### iteration step :  70 rmse : 0.20685455302331512
### iteration step :  80 rmse : 0.19413418783028674
### iteration step :  90 rmse : 0.1847008200272031
### iteration step :  100 rmse : 0.17742927527209082
### iteration step :  110 rmse : 0.17165226964707506
### iteration step :  120 rmse : 0.16695181946871496
### iteration step :  130 rmse : 0.16305292191997453
### iteration step :  140 rmse : 0.159766919296796
### iteration step :  150 rmse : 0.15695986999457337
### iteration step :  160 rmse : 0.15453398186715442
### iteration step :  170 rmse : 0.1524161855107769
### iteration step :  180 rmse : 0.1505508073962834
### iteration

직관적으로 보기 위해 칼럼명 영화 타이틀로 변경!

In [ ]:
ratings_pred_matrix = pd.DataFrame(data = pred_matrix, index=ratings_matrix.index,
                                   columns = ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.055084,4.092018,3.564130,4.502167,3.981215,1.271694,3.603274,2.333266,5.091749,3.972454,...,1.402608,4.208382,3.705957,2.720514,2.787331,3.475076,3.253458,2.161087,4.010495,0.859474
2,3.170119,3.657992,3.308707,4.166521,4.311890,1.275469,4.237972,1.900366,3.392859,3.647421,...,0.973811,3.528264,3.361532,2.672535,2.404456,4.232789,2.911602,1.634576,4.135735,0.725684
3,2.307073,1.658853,1.443538,2.208859,2.229486,0.780760,1.997043,0.924908,2.970700,2.551446,...,0.520354,1.709494,2.281596,1.782833,1.635173,1.323276,2.887580,1.042618,2.293890,0.396941


In [ ]:
# 사용자 9번 영화추천

def get_unseen_movies(ratings_matrix, userId):
    # userId로 입력받은 사용자의 모든 영화 정보를 추출해 Series로 반환함.
    # 반환된 user_rating은 영화명(title)을 인덱스를 가지는 Series 객체임.
    user_rating = ratings_matrix.loc[userId, :]

    #user_rating이 0보다 크면 기존에 관람한 영화임. 대상 인덱스를 추출해 list 객체로 만듦.
    already_seen = user_rating[ user_rating > 0].index.tolist()

    # 모든 영화명을 list 객체로 만듦.
    movies_list = ratings_matrix.columns.tolist()

    # list comporehension으로 already_seen에 해당하는 영화는 movies_list에서 제외함.
    unseen_list = [ movie for movie in movies_list if movie not in already_seen]

    return unseen_list

def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n = 10):
    # 예측 평점 DataFrame에서 사용자id 인덱스와 unseen_list로 들어온 영화명 칼럼을 추출해
    # 가장 예측 평점이 높은 순으로 정렬함.
    recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]
    return recomm_movies

> 개인화된 영화 추천 실습



In [ ]:
# 사용자가 관람하지 않은 영화명 추출
unseen_list = get_unseen_movies(ratings_matrix, 9)

# 잠재 요인 협업 필터링으로 영화 추천
recomm_movies = recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n = 10)

# 평점 데이터를 DataFrame으로 생성.
recomm_movies = pd.DataFrame(data = recomm_movies.values, index = recomm_movies.index,
                             columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704612
"South Park: Bigger, Longer and Uncut (1999)",5.451100
Rounders (1998),5.298393
Blade Runner (1982),5.244951
Roger & Me (1989),5.191962
Gattaca (1997),5.183179
Ben-Hur (1959),5.130463
Rosencrantz and Guildenstern Are Dead (1990),5.087375
"Big Lebowski, The (1998)",5.038690


(앞 절) 아이템 기반 협업 필터링 결과와는 많이 다른 라인업 ㅎ

## 9-08 파이썬 추천 시스템 패키지 一 Surprise

### Surprise 패키지 소개

파이선 기반 추천시스템 구축을 위한 전용 패키지.  
- 다양한 추천 알고리즘 쉽게 적용해 구축 가능.
- 사이킷런과 유사한 API, 프레임워크 제공.

In [ ]:
!pip install scikit-surprise

### Surprise를 이용한 추천 시스템 구축

Surprise에서 데이터 로딩
-  Dataset 클래스를 이용해서만 가능
- 주요 데이터가 Row 레벨 형태로 되어 있는 포맷의 데이터만 처리.


` load_builtin()`: 무비렌즈 사이트에서 제공하는 과거 버전 데이터 세트 가져오는 API  


In [ ]:
!pip install "numpy<2.0.0" scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.7 MB/s eta 0:00:00
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554990 sha256=dc869b3b15912698856cb4a141cec79127f3dc746d29a4728cc6436f5a788bfc
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following 

In [ ]:
import numpy as np
from surprise import SVD, Dataset, accuracy
from surprise.model_selection import train_test_split

> 데이터 로딩 후 학습/테스트 데이터 세트로 분리

In [ ]:
data = Dataset.load_builtin('ml-100k')
# 수행 시마다 동일하게 데이터를 분할하기 위해 random_state 값 부여
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


한 번 로컬 디렉터리에 데이터 저장된 후에는 호출하면 됨,  
최근 영화 평점 정보가 아님!  
분리문자가 (,)가 아니고 (\t) 문자임

주의) 로우 레벨 데이터 적용해야함  

> SVD로 잠재 요인 협업 필터링 수행

In [ ]:
algo = SVD(random_state=0)  # 알고리즘 객체 생성
algo.fit(trainset)

> 학습된 추천 알고리즘 기반으로 추천 수행

- `test()`:  사용자-아이템 평점 데이터 세트 전체에 대해서 추천을 예측하는 메서드
- `predict()`: 개별 사용자와 영화에 대한 추천 평점을 반환

In [ ]:
predictions = algo.test(testset)
print('prediction type:', type(predictions), 'size: ', len(predictions))
print('prediction 결과의 최초 5개 추출')
predictions[:5]

prediction type: <class 'list'> size:  25000
prediction 결과의 최초 5개 추출


[Prediction(uid='120', iid='282', r_ui=4.0, est=3.5114147666251547, details={'was_impossible': False}),
 Prediction(uid='882', iid='291', r_ui=4.0, est=3.573872419581491, details={'was_impossible': False}),
 Prediction(uid='535', iid='507', r_ui=5.0, est=4.033583485472447, details={'was_impossible': False}),
 Prediction(uid='697', iid='244', r_ui=5.0, est=3.8463639495936905, details={'was_impossible': False}),
 Prediction(uid='751', iid='385', r_ui=4.0, est=3.1807542478219157, details={'was_impossible': False})]

파이썬 리스트 / 입력 인자 데이터 세트 크기와 동일한 크기.  

'was_impossible'=True이면 예측값 생성 불가 데이터라는 뜻

객체명.uid 식으로 속성 접근 및 추출 가능

In [ ]:
[ (pred.uid, pred.iid, pred.est) for pred in predictions[:3]]

[('120', '282', 3.5114147666251547),
 ('882', '291', 3.573872419581491),
 ('535', '507', 4.033583485472447)]

> predict()로 추천 예측 ㄱㄱ

개별 사용자 아이디, 아이템 아이디 입력 -> 추천 예측 평점 포함한 정보 반환

In [ ]:
# 사용자 아이디, 아이템 아이디는 문자열로 입력해야함
uid = str(196)
iid = str(302)
pred = algo.predict(uid, iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}


추천 예측 평점을 est로 반환

test()는 입력 데이터 세트의 모든 사용자와 아이템 아이디에 대해서 predict()를 반복적으로
수행한 결과st...

> 추천 예측 평점 - 실제 평점 차이 평가

accuracy 모듈 사용

In [ ]:
accuracy.rmse(predictions)

RMSE: 0.9467


0.9466860806937948

### Surprise 주요 모듈 소개

#### Dataset

 데이터 세트의 칼럼 순서가 사용자 아이디, 아이템 아이디, 평점 순 정렬되어야만 함

#### OS 파일 데이터를 Surprise 데이터 세트로 로딩

주의)  로딩되는 데이터 파일에 칼럼명을 가지는 헤더 문자열이 있어서는 안 됨!!






In [ ]:
import pandas as pd

ratings = pd.read_csv('/content/ratings.csv')
# ratings_noh.csv 파일로 언로드 시 인덱스와 헤더를 모두 제거한 새로운 파일 생성
ratings.to_csv('/content/ratings.csv', index=False, header=False)

>  DataSet 모듈의 load_from_file( )을 이용해 DataSet로 로드

Reader 클래스: 데이터 파일의 파싱 포맷을 정의

칼럼 헤더가 없고, 4개의 칼럼이 콤마로만 분리돼 있으므로 로딩할 때 칼럼이 뭔지 알려줘야됨

In [ ]:
from surprise import Reader
reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5))
data = Dataset.load_from_file('/content/ratings.csv', reader=reader)

> SVD 행렬 분해 기법 이용해 추천 예측

In [ ]:
trainset, testset = train_test_split(data, test_size = .25, random_state=0)

# 수행 시마다 동일한 결과를 도출하기 위해 random_state 설정
algo = SVD(n_factors=50, random_state=0)

# 학습 데이터 세트로 학습하고 나서 테스트 데이터 세트로 평점 예측 후 RMSE 평가
algo.fit(trainset)
predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.8682


0.8681952927143516

#### 판다스 DataFrame에서 Surprise 데이터 세트로 로딩

`Dataset.load_from_df( )` 이용하면 가능.  
여기서도 칼럼 순서 지켜야됨

In [ ]:
import pandas as pd
from surprise import Reader, Dataset

col_names = ['userId', 'movieId', 'rating', 'timestamp']
ratings = pd.read_csv('/content/ratings.csv', names=col_names, header=None)
reader = Reader(rating_scale = (0.5, 5.0))

# ratings DataFrame에서 칼럼은 사용자 아이디, 아이템 아이디, 평점 순서를 지켜야 함!! 파라미터 순서대로 입력
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

algo = SVD(n_factors = 50, random_state=0)
algo.fit(trainset)
predictions = algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.8682


0.8681952927143516

### Surprise 추천 알고리즘 클래스


- SVD
- KNNBasic
- BaselineOnly


### 베이스라인 평점

개인의 성향에 따라 같은 아이템이더라도 평가가 달라질 수 있음   
-> 개인의 성향을 반영해 아이템 평가에 편향성(bias) 요소를 반영하여 평점을 부과하는 것

전체 평균 평점 + 사용자 편향 점수 + 아이템 편향 점수 공식으로 계산

- 전체 평균 평점 = 모든 사용자의 아이템에 대한 평점을 평균한 값
- 사용자 편향 점수=사용자별 아이템 평점 평균 값 - 전체 평균 평점
- 아이템 편향 점수=아이템별 평점 평균 값 - 전체 평균 평점


### 교차 검증과 하이퍼 파라미터 튜닝

cross_validate( )와 GridSearchCV 클래스를 제공

> cross_validate( )

인자로  알고리즘 객체, 데이터, 성능 평가 방법(measures), 폴드 데이터 세트 개수(cv) 입력

In [ ]:
from surprise.model_selection import cross_validate

# 판다스 DataFrame에서 Surprise 데이터 세트로 데이터 로딩
col_names = ['userId', 'movieId', 'rating', 'timestamp']
ratings = pd.read_csv('/content/ratings.csv', names=col_names, header=None) # reading data in pandas df
reader = Reader(rating_scale = (0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

algo = SVD(random_state=0)
cross_validate(algo, data, measures = ['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8735  0.8692  0.8672  0.8835  0.8733  0.8733  0.0056  
MAE (testset)     0.6698  0.6678  0.6679  0.6786  0.6689  0.6706  0.0041  
Fit time          2.80    1.38    1.41    1.36    1.70    1.73    0.55    
Test time         0.31    0.10    0.30    0.11    0.18    0.20    0.09    


{'test_rmse': array([0.87353989, 0.86917136, 0.86723889, 0.88353702, 0.8732599 ]),
 'test_mae': array([0.66976491, 0.66777225, 0.66794529, 0.67864352, 0.66886352]),
 'fit_time': (2.7981200218200684,
  1.3762133121490479,
  1.4098854064941406,
  1.3639090061187744,
  1.6986870765686035),
 'test_time': (0.3111414909362793,
  0.10399317741394043,
  0.30324363708496094,
  0.11304068565368652,
  0.17957520484924316)}

 폴드별 성능 평가 수치와 전체 폴드의 평균 성능 평가 수치를 함께 보여줌

 >  GridSearchCV

교차 검증을 통한 하이퍼 파라미터 최적화를 수행  

SVD: 주로 n_epochs, n_factors 튜닝


In [ ]:
from surprise.model_selection import GridSearchCV

# 최적화할 파라미터를 딕셔너리 형태로 지정
param_grid = {'n_epochs': [20, 40, 60], 'n_factors': [50, 100, 200]}

# CV를 3개 폴드 세트로 지정, 성능 평가는 rmse, mse로 수행하도록 GridSearchCV 구성
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

# 최고 RMSE Evaluation 점수와 그때의 하이퍼 파라미터
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.878787763829702
{'n_epochs': 20, 'n_factors': 50}


### Surprise를 이용한 개인화 영화 추천 시스템 구축

Surprise 패키지로 학습된 추천 알고리즘을 기반으로 특정 사용자가 아직 평점을 매기지 않은(관람하지 않은) 영화 중에서 개인 취향에 가장 적절한 영화를 추천

학습/테스트 데이터로 분리X, 전체를 학습 데이터로 사용.

BUT  Surprise는 데이터 세트를 train_test_split()을 이용해 내부에서 사용하는 TrainSet 클래스 객체로 변환하지 않으면 fit()을 통해 학습 불가

In [ ]:
# 다음 코드는 train_test_split()으로 분리되지 않는 데이터 세트에 fit()을 호출해 오류가 발생합니다.
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
algo = SVD(n_factors=50, random_state=0)
algo.fit(data)

AttributeError: 'DatasetAutoFolds' object has no attribute 'n_users'

-> DatasetAutoFolds 클래스를 이용

> DatasetAutoFolds 객체 생성 후  build_full_trainset() 메서드 호출

In [ ]:
from surprise.dataset import DatasetAutoFolds

reader = Reader(line_format = 'user item rating timestamp', sep=',', rating_scale = (0.5, 5))
# DatasetAutoFolds 클래스를 ratings_noh.csv 파일 기반으로 생성
data_folds = DatasetAutoFolds(ratings_file='/content/ratings.csv', reader = reader)

# 전체 데이터를 학습 데이터로 생성함
trainset = data_folds.build_full_trainset()

In [ ]:
algo = SVD(n_epochs=20, n_factors=50, random_state=0)
algo.fit(trainset)

In [ ]:
# 영화에 대한 상세 속성 정보 DataFrame 로딩
movies = pd.read_csv('/content/movies.csv')

# userId = 9의 movieId 데이터를 추출해 movieId = 42 데이터가 있는지 확인
movieIds = ratings[ratings['userId']==9]['movieId']
if movieIds[movieIds==42].count() ==0:
  print('사용자 아이디 9는 영화 아이디 42의 평점 없음')

print(movies[movies['movieId']==42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
    movieId                   title              genres
38       42  Dead Presidents (1995)  Action|Crime|Drama


> 추천 예상 평점

predict() 메서드 이용

In [ ]:
uid = str(9)
iid = str(42)

pred = algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 3.13   {'was_impossible': False}


> 사용자가 평점을 매기지 않은 영화의 추천 예측 평점 도출

In [ ]:
def get_unseen_surprise(ratings, movies, userId):
  # 입력값으로 들어온 userId에 해당하는 사용자가 평점을 매긴 모든 영화를 리스트로 생성
  seen_movies = ratings[ratings['userId']==userId]['movieId'].tolist()

  # 모든 영화의 movieId를 리스트로 생성
  total_movies = movies['movieId'].tolist()

  # 모든 영화의 movieId 중 이미 평점을 매긴 영화의 movieId를 제외한 후 리스트로 생성
  unseen_movies = [movie for movie in total_movies if movie not in seen_movies]
  print('평점 매긴 영화 수:', len(seen_movies), '추천 대상 영화 수:', len(unseen_movies),
        '전체 영화 수:', len(total_movies))

  return unseen_movies

unseen_movies = get_unseen_surprise(ratings, movies, 9)

평점 매긴 영화 수: 46 추천 대상 영화 수: 9696 전체 영화 수: 9742


> SVD 이용해 높은 예측 평점 가진 순으로 영화 추천

In [ ]:
def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):

  # 알고리즘 객체의 predict() 메서드를 평점이 없는 영화에 반복 수행한 후 결과를 list 객체로 저장
  predictions = [algo.predict(str(userId), str(movieId)) for movieId in unseen_movies]

  # predictions list 객체는 surprise의 predictions 객체를 원소로 가지고 있음
  # [Prediction(uid = '9', iid = '1', est=3.69), Prediction(uid='9', iid='2', est=2.98),,,]

  # 이를 est 값으로 정렬하기 위해서 아래의 sortkey_est 합수를 정의함
  # sortkey_est 함수는 list 객체의 sort() 함수의 키 값으로 사용되어 정렬 수행
  def sortkey_est(pred):
    return pred.est

  # sortkey_est() 반환값의 내림차순으로 정렬 수행하고 top_n개의 최상위 값 추출
  predictions.sort(key=sortkey_est, reverse=True)
  top_predictions = predictions[:top_n]

  # top_n으로 추출된 영화의 정보 추출. 영화 아이디, 추천 예상 평점, 제목 추출
  top_movie_ids = [int(pred.iid) for pred in top_predictions]
  top_movie_rating = [pred.est for pred in top_predictions]
  top_movie_titles = movies[movies.movieId.isin(top_movie_ids)]['title']
  top_movie_preds = [(id, title, rating) for id, title, rating in \
                     zip(top_movie_ids, top_movie_titles, top_movie_rating)]

  return top_movie_preds

unseen_movies = get_unseen_surprise(ratings, movies, 9)
top_movie_preds = recomm_movie_by_surprise(algo, 9, unseen_movies, top_n=10)

print('##### Top-10 추천 영화 리스트 #####')
for top_movie in top_movie_preds:
  print(top_movie[1], ':', top_movie[2])

평점 매긴 영화 수: 46 추천 대상 영화 수: 9696 전체 영화 수: 9742
##### Top-10 추천 영화 리스트 #####
Usual Suspects, The (1995) : 4.306302135700814
Star Wars: Episode IV - A New Hope (1977) : 4.281663842987387
Pulp Fiction (1994) : 4.278152632122759
Silence of the Lambs, The (1991) : 4.226073566460876
Godfather, The (1972) : 4.1918097904381995
Streetcar Named Desire, A (1951) : 4.154746591122657
Star Wars: Episode V - The Empire Strikes Back (1980) : 4.122016128534504
Star Wars: Episode VI - Return of the Jedi (1983) : 4.108009609093436
Goodfellas (1990) : 4.083464936588478
Glory (1989) : 4.07887165526957


## 9-09 정리

추천시스템
- 콘텐츠 기반 필터링
- 협업 필터링
  - 잠재 요인 협업 필터링
- Surprise